# UIT-ViQuAD2.0 — Qwen2.5-3B: train LoRA / DoRA / TinyLoRA

Notebook extractive QA (SQuAD 2.0) tối ưu **score + thời gian train**:
- Prompt ép **chỉ trả span** (không câu dài)
- **Post-process**: bóc prefix + align prediction về span trong context khi eval
- **Early stopping**: dừng khi `eval_loss` (validation) không giảm — `MAX_TRAIN_EPOCHS` chỉ là trần
- **LoRA r=16, alpha=32** (parity 1.5B; đủ cho MRC, nhanh hơn r=32)
- **bf16** trên 3090/4090 (`FORCE_FP16=False`) — ổn định hơn fp16
- **Attention nhanh**: thử `flash_attention_2` → fallback `sdpa` (xformers/Unsloth)
- **adamw_8bit** + batch nhỏ + grad accum
- **Chọn adapter**: `TRAIN_METHODS = ["lora"]` hoặc `["dora"]` / `["tinylora"]` / cả 3
- **Resume**: `RESUME_FROM_CHECKPOINT = True` (checkpoint mỗi 200 steps)

**GPU mục tiêu: RTX 4090 / 3090 ~24GB**

| Preset | batch | grad_accum | effective | Ghi chú |
|--------|-------|------------|-----------|---------|
| **Speed (mặc định)** | 4 | 2 | 8 | Nhanh trên 4090/3090 + FA2/SDPA |
| Safe | 2 | 4 | 8 | Nếu OOM |
| OOM fallback | 1 | 8 | 8 | An toàn VRAM |
| QLoRA | 2 | 4 + `LOAD_IN_4BIT=True` | 8 | Tiết kiệm VRAM |

### Cách chạy
1. Pip cells → **Restart Kernel**
2. Warnings → Config → Data → Profiling → Dataset format
3. Chỉnh `TRAIN_METHODS` (train từng loại) → **Train**
4. **Evaluation**


In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# TinyLoRA cần PEFT >= 0.19 (API: r, u, tinylora_dropout — không có lora_alpha)
!pip install "peft>=0.19.0" transformers trl accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn --no-cache-dir
# Optional FlashAttention-2 (thường OK trên Linux/Colab). Build fail → notebook tự fallback SDPA.
import subprocess, sys
_r = subprocess.run([sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation", "--no-cache-dir"])
print("flash-attn OK" if _r.returncode == 0 else "[WARN] flash-attn install failed — will use sdpa")


In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import TinyLoraConfig

print(f"PEFT version: {peft.__version__}")
sig = inspect.signature(TinyLoraConfig.__init__).parameters
print("TinyLoraConfig params:", ", ".join(k for k in sig if k != "self"))

In [ ]:
import warnings, logging, os, sys
import torch

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
# Tránh download HF bị treo 30p+ (XET stall → retry chậm)
os.environ["HF_HUB_DISABLE_XET"] = "1"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# RTX 3090 (Ampere) — bật TF32 + cuDNN autotune để tăng throughput matmul/conv
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print("Warnings suppressed.")
if torch.cuda.is_available():
    print(f"CUDA speed: TF32={torch.backends.cuda.matmul.allow_tf32}, cudnn.benchmark={torch.backends.cudnn.benchmark}")

In [ ]:
import gc
import json
import math
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    cc = torch.cuda.get_device_capability(0)
    arch = "Ampere (3090)" if cc == (8, 6) else "Ada (4090)" if cc == (8, 9) else f"sm_{cc[0]}{cc[1]}"
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | {arch}")
    if cc == (8, 6):
        print("-> RTX 3090: bf16 + TF32 + batch=4/grad_accum=2 + FA2/SDPA")
    elif cc == (8, 9):
        print("-> RTX 4090: bf16 + TF32 + batch=4/grad_accum=2 + FA2/SDPA")

# ========== CẤU HÌNH 3B ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

# Prompt siết chặt → ép model trả span-only (tăng EM)
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm bất kỳ từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:', 'là', 'thủ đô'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn dưới đây. "
    "Chỉ trả về cụm từ trong đoạn văn, không thêm từ nào khác.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

EVAL_RESULTS_PATH = "eval_results_viquad2_3b.json"
COMPARE_EVAL_PATH = "eval_compare_adapters_viquad2_3b.json"
PROFILING_CONFIG_PATH = "profiling_config_3b.json"

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"

RUN_TRAINING = True
RUN_EVALUATION = True
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

# ========== RTX 4090/3090 ~24GB — SPEED + SCORE PRESET ==========
# LoRA full precision bf16 (không 4-bit) — đủ VRAM với max_seq_length ~1024
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
FORCE_FP16 = False         # False = bf16 trên 3090/4090 (ổn định hơn fp16)
# Attention: "auto" = flash_attention_2 nếu có, không thì sdpa
ATTN_IMPLEMENTATION = "auto"
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32
USE_SPAN_POSTPROCESS = True

# Speed preset: batch=4/accum=2 (effective=8). OOM -> (2,4) hoặc (1,8)
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# LoRA/DoRA rank — parity với 1.5B (r=16, alpha=32)
LORA_R = 16
LORA_ALPHA = 32

# Early stopping — dừng khi eval_loss (validation) không cải thiện
USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS = 5              # trần epoch tối đa (thay vì cố định 3)
EARLY_STOPPING_PATIENCE = 3         # số lần eval liên tiếp không giảm loss
EARLY_STOPPING_THRESHOLD = 0.001  # cải thiện tối thiểu để tính là "tốt hơn"
EVAL_STEPS = 200                    # eval validation mỗi N steps (= SAVE_STEPS)

TRAIN_COMMON = dict(
    per_device_train_batch_size=4,   # 4090/3090 + FA2/SDPA: nhanh hơn batch=2
    gradient_accumulation_steps=2,  # effective batch = 8; OOM -> batch=2, accum=4
    warmup_steps=10,
    num_train_epochs=MAX_TRAIN_EPOCHS,
    learning_rate=1e-4,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

# Preprocessing / dataloader — workers=0 an toàn nhất trên Jupyter
DATASET_NUM_PROC = 4
DATALOADER_NUM_WORKERS = 0

# Checkpoint / resume — mất mạng: đặt RESUME_FROM_CHECKPOINT = True
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 3
RESUME_FROM_CHECKPOINT = False  # True | "outputs_viquad2_3b_lora/checkpoint-1200"

# Chọn adapter cần train / eval
ADAPTER_VARIANTS = [
    {"name": "lora", "save_path": "qwen2.5-3b-instruct-lora-viquad2", "output_dir": "outputs_viquad2_3b_lora"},
    {"name": "dora", "save_path": "qwen2.5-3b-instruct-dora-viquad2", "output_dir": "outputs_viquad2_3b_dora"},
    {"name": "tinylora", "save_path": "qwen2.5-3b-instruct-tinylora-viquad2", "output_dir": "outputs_viquad2_3b_tinylora"},
]
TRAIN_METHODS = ["lora"]  # hoặc ["lora", "dora", "tinylora"]
EVAL_METHODS = ["lora"]   # method nào có adapter sẽ được eval

TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"

def resolve_attn_implementation(preferred=ATTN_IMPLEMENTATION):
    """flash_attention_2 nếu đã cài; không thì sdpa (nhanh trên Ampere/Ada)."""
    if preferred and preferred != "auto":
        return preferred
    try:
        import flash_attn  # noqa: F401
        return "flash_attention_2"
    except Exception:
        return "sdpa"


def filter_training_args(kwargs):
    """Tương thích transformers mới/cũ: eval_strategy vs evaluation_strategy."""
    import inspect
    from transformers import TrainingArguments

    sig = inspect.signature(TrainingArguments.__init__).parameters
    out = dict(kwargs)
    # Rename legacy -> new
    if "evaluation_strategy" in out and "eval_strategy" in sig and "evaluation_strategy" not in sig:
        out["eval_strategy"] = out.pop("evaluation_strategy")
    elif "eval_strategy" in out and "evaluation_strategy" in sig and "eval_strategy" not in sig:
        out["evaluation_strategy"] = out.pop("eval_strategy")
    # Drop unknown keys (avoid TypeError on version skew)
    return {k: v for k, v in out.items() if k == "self" or k in sig}



def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def get_variant(method_name):
    for variant in ADAPTER_VARIANTS:
        if variant["name"] == method_name:
            return variant
    raise ValueError(f"Unknown adapter method: {method_name}")


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    # Qwen tokenizer đôi khi pad_token=None → dễ lỗi ở generate / trainer
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


In [ ]:
def row_to_sample(row):
    is_impossible = bool(row["is_impossible"])
    texts = (row.get("answers") or {}).get("text") or []
    if is_impossible or len(texts) == 0:
        return {
            "id": row.get("id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True,
        }
    answer = texts[0]
    if not str(answer).strip():
        return {
            "id": row.get("id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True,
        }
    return {
        "id": row.get("id", ""), "title": row.get("title", ""),
        "context": row["context"], "question": row["question"],
        "answer": answer, "is_impossible": False,
    }


def hf_split_to_samples(split_dataset):
    return [row_to_sample(row) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(samples)} → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"])
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"Không tải được HF: {e}")
    with open(TEST_JSON_PATH, encoding="utf-8") as f: test_samples = json.load(f)
    with open(DEV_JSON_PATH, encoding="utf-8") as f: dev_samples = json.load(f)
    train_samples = dev_samples

split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)

In [ ]:
if "train_samples" not in globals():
    raise NameError("Chạy cell tải dataset trước.")


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(s, tok))))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {"min": lengths[0], "p50": lengths[n//2], "p95": lengths[int(n*0.95)],
             "p99": lengths[int(n*0.99)], "max": lengths[-1]}
    chosen = max(((min(math.ceil(stats["p99"]*1.05), cap)+255)//256)*256, min_len)
    stats.update({"chosen_max_seq_length": chosen,
                  "truncated_samples": sum(1 for L in lengths if L > chosen),
                  "truncated_pct": round(100*sum(1 for L in lengths if L > chosen)/n, 3)})
    return chosen, stats


tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    prof_cfg = json.load(open(PROFILING_CONFIG_PATH, encoding="utf-8"))
    max_seq_length, length_stats = prof_cfg["max_seq_length"], prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats},
              open(PROFILING_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"max_seq_length = {max_seq_length}")
for k, v in length_stats.items(): print(f"  {k}: {v}")

In [ ]:
# Format dataset 1 lần — dùng chung cho mọi adapter type
from datasets import Dataset

tokenizer_fmt = load_tokenizer()

def formatting_prompts_func(examples):
    texts = []
    for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
        msgs = build_messages(ctx, q, ans)
        texts.append(tokenizer_fmt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

train_hf = Dataset.from_list(train_samples)
try:
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        batch_size=256,
        num_proc=DATASET_NUM_PROC,
        remove_columns=train_hf.column_names,
    )
except Exception as e:
    print(f"num_proc={DATASET_NUM_PROC} failed ({e}) → fallback num_proc=1")
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=train_hf.column_names,
    )
print(f"Shared train dataset: {len(dataset)} samples")

eval_dataset = None
if USE_EARLY_STOPPING:
    dev_hf = Dataset.from_list(dev_samples)
    try:
        eval_dataset = dev_hf.map(
            formatting_prompts_func,
            batched=True,
            batch_size=256,
            num_proc=DATASET_NUM_PROC,
            remove_columns=dev_hf.column_names,
        )
    except Exception as e:
        print(f"Eval map num_proc failed ({e}) → fallback num_proc=1")
        eval_dataset = dev_hf.map(
            formatting_prompts_func,
            batched=True,
            remove_columns=dev_hf.column_names,
        )
    print(f"Eval dataset (early stopping): {len(eval_dataset)} samples")

## Train adapter(s) — LoRA / DoRA / TinyLoRA (RTX 4090/3090)

Mỗi method: load base → gắn adapter → train (checkpoint mỗi `SAVE_STEPS`) → lưu adapter → giải phóng GPU.

**Speed preset:** `batch=4`, `grad_accum=2`, **bf16**, FA2/SDPA, Unsloth gradient checkpointing, `r=16/alpha=32`.

- Chỉnh `TRAIN_METHODS` để train **từng loại** (`["lora"]` / `["dora"]` / `["tinylora"]`)
- **OOM:** giảm `per_device_train_batch_size` (4→2→1), tăng `gradient_accumulation_steps` tương ứng
- Mất mạng / crash: `RESUME_FROM_CHECKPOINT = True` rồi chạy lại cell train
- Lỗi `evaluation_strategy`: đã tự map → `eval_strategy` (transformers mới)


In [ ]:
def resolve_resume_checkpoint(output_dir, resume_flag):
    """Trả về path checkpoint để resume, hoặc None nếu train từ đầu."""
    output_dir = Path(output_dir)
    if not resume_flag:
        return None
    if isinstance(resume_flag, str):
        ckpt = Path(resume_flag)
        if not ckpt.exists():
            raise FileNotFoundError(f"Checkpoint không tồn tại: {ckpt}")
        return str(ckpt)
    ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.rsplit("-", 1)[-1]))
    return str(ckpts[-1]) if ckpts else None


def apply_adapter(model, method_name):
    """Gắn LoRA / DoRA / TinyLoRA lên base model đã load bằng Unsloth."""
    from unsloth import FastLanguageModel

    def _resolve_causallm(m):
        if hasattr(m, "prepare_inputs_for_generation"):
            return m
        if hasattr(m, "get_base_model"):
            base = m.get_base_model()
            if hasattr(base, "prepare_inputs_for_generation"):
                return base
        raise RuntimeError(
            "Không tìm thấy ForCausalLM. Đừng dùng model.model — đó là Qwen2Model "
            "(thiếu prepare_inputs_for_generation)."
        )

    if method_name == "lora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=False,
        )
        print(f"Applied: LoRA (r={LORA_R}, alpha={LORA_ALPHA})")
        return model

    if method_name == "dora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=True,
        )
        print(f"Applied: DoRA (r={LORA_R}, alpha={LORA_ALPHA}, use_dora=True)")
        return model

    if method_name == "tinylora":
        import inspect
        import peft
        from peft import TinyLoraConfig, get_peft_model as peft_get_model

        sig = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {
            "r": 2,
            "u": 64,
            "num_projections": 64,
            "target_modules": TARGET_MODULES,
            "tinylora_dropout": 0.0,
            "lora_dropout": 0.0,
            "bias": "none",
            "task_type": "CAUSAL_LM",
            "weight_tying": 0.0,
            "projection_seed": 3407,
            "init_weights": True,
        }
        tiny_kwargs = {k: v for k, v in desired.items() if k in sig}
        if "target_modules" not in tiny_kwargs:
            tiny_kwargs["target_modules"] = TARGET_MODULES

        try:
            tinylora_config = TinyLoraConfig(**tiny_kwargs)
        except Exception as e:
            raise RuntimeError(
                f"TinyLoRA không khả dụng với PEFT {peft.__version__}: {e}. "
                "Chạy lại pip cell (peft>=0.19) rồi Restart Kernel."
            ) from e

        causallm = _resolve_causallm(model)
        model = peft_get_model(causallm, tinylora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        u_val = tiny_kwargs.get("u", tiny_kwargs.get("num_projections", "?"))
        print(f"Applied: TinyLoRA (PEFT {peft.__version__}, r={tiny_kwargs.get('r')}, u={u_val})")
        return model

    raise ValueError(f"Unknown method: {method_name}")


def _resolve_load_and_train_dtype():
    """Chọn dtype load + fp16/bf16 train — phải khớp nhau (Unsloth strict)."""
    import torch
    from unsloth import is_bfloat16_supported

    if LOAD_IN_4BIT or LOAD_IN_8BIT:
        load_dtype = None
        use_bf16 = is_bfloat16_supported() and not FORCE_FP16
        use_fp16 = not use_bf16
    elif FORCE_FP16 or not is_bfloat16_supported():
        load_dtype = torch.float16
        use_fp16, use_bf16 = True, False
    else:
        load_dtype = None
        use_fp16, use_bf16 = False, True
    return load_dtype, use_fp16, use_bf16


def _sync_train_dtype_with_model(model, use_fp16, use_bf16):
    """Đọc dtype thực tế của model sau load/adapter — tránh TypeError Unsloth."""
    import torch

    param_dtype = next((p.dtype for p in model.parameters() if p.requires_grad or p.numel() > 0), None)
    if param_dtype == torch.float16:
        return True, False
    if param_dtype == torch.bfloat16:
        return False, True
    return use_fp16, use_bf16


def train_one_variant(variant, max_seq_length, dataset, eval_dataset=None, resume_flag=RESUME_FROM_CHECKPOINT):
    import torch
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    from transformers import TrainingArguments, EarlyStoppingCallback
    import sys

    name = variant["name"]
    print("\n" + "=" * 60)
    print(f" TRAIN VARIANT: {name.upper()}")
    print("=" * 60)

    eval_on = USE_EARLY_STOPPING and eval_dataset is not None
    if USE_EARLY_STOPPING and eval_dataset is None:
        print("⚠ USE_EARLY_STOPPING=True nhưng thiếu eval_dataset — train tới MAX_TRAIN_EPOCHS")

    clear_gpu()
    load_dtype, use_fp16, use_bf16 = _resolve_load_and_train_dtype()
    print(f"Dtype plan: load={load_dtype or 'auto'} | train={'fp16' if use_fp16 else 'bf16'}")
    print(f"Loading base model: {BASE_MODEL_NAME} (lần đầu có thể tải ~6GB, 5–15 phút)...", flush=True)

    attn_impl = resolve_attn_implementation()
    print(f"Attention: {attn_impl}", flush=True)
    load_kwargs = dict(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=load_dtype,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
        attn_implementation=attn_impl,
    )
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    except Exception as e:
        if attn_impl != "sdpa":
            print(f"[WARN] attn={attn_impl} failed ({e}); fallback sdpa", flush=True)
            load_kwargs["attn_implementation"] = "sdpa"
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        else:
            raise
    print("Base model loaded.", flush=True)
    model = apply_adapter(model, name)

    use_fp16, use_bf16 = _sync_train_dtype_with_model(model, use_fp16, use_bf16)
    print(f"Dtype verified: train={'fp16' if use_fp16 else 'bf16'}")

    train_args = dict(
        **TRAIN_COMMON,
        fp16=use_fp16,
        bf16=use_bf16,
        output_dir=variant["output_dir"],
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=SAVE_STEPS,
        logging_first_step=False,
        log_level="error",
        log_level_replica="error",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=True,
    )
    callbacks = []
    if eval_on:
        train_args.update(
            eval_strategy="steps",  # transformers mới; filter_training_args map legacy
            evaluation_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        )
        callbacks.append(EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ))
        print(
            f"Early stopping ON: max_epochs={MAX_TRAIN_EPOCHS}, patience={EARLY_STOPPING_PATIENCE}, "
            f"eval every {EVAL_STEPS} steps on {len(eval_dataset)} val samples"
        )
    else:
        train_args["eval_strategy"] = "no"
        train_args["evaluation_strategy"] = "no"

    train_args = filter_training_args(train_args)
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=dataset,
        eval_dataset=eval_dataset if eval_on else None,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=DATASET_NUM_PROC,
        packing=False,
        args=TrainingArguments(**train_args),
        callbacks=callbacks,
    )
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls

    effective_batch = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps
    steps_per_epoch = math.ceil(len(dataset) / max(effective_batch, 1))
    total_steps = int(steps_per_epoch * trainer.args.num_train_epochs)
    print(
        f"Planned: batch={trainer.args.per_device_train_batch_size}×accum={trainer.args.gradient_accumulation_steps} "
        f"(effective={effective_batch}) | max_epochs={trainer.args.num_train_epochs} | "
        f"steps/epoch≈{steps_per_epoch} | total_steps≈{total_steps}"
        + (f" | early_stop patience={EARLY_STOPPING_PATIENCE}" if eval_on else "")
    )

    resume_ckpt = resolve_resume_checkpoint(variant["output_dir"], resume_flag)
    if resume_ckpt:
        print(f"Resume from checkpoint: {resume_ckpt}")
        trainer.train(resume_from_checkpoint=resume_ckpt)
    elif resume_flag:
        print("RESUME_FROM_CHECKPOINT=True nhưng chưa có checkpoint — train từ đầu.")
        trainer.train()
    else:
        trainer.train()

    Path(variant["save_path"]).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(variant["save_path"])
    tokenizer.save_pretrained(variant["save_path"])
    print(f"Saved adapter → {variant['save_path']}")

    del trainer, model, tokenizer
    clear_gpu()
    return variant["save_path"]


In [ ]:
if RUN_TRAINING:
    variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map:
            raise ValueError(f"Unknown TRAIN_METHODS item: {method}")
        path = train_one_variant(
            variant_map[method], max_seq_length, dataset,
            eval_dataset=eval_dataset if USE_EARLY_STOPPING else None,
        )
        trained_paths[method] = path
    print("\n✅ Train xong:")
    for k, v in trained_paths.items():
        print(f"  {k}: {v}")
else:
    trained_paths = {m: get_variant(m)["save_path"] for m in EVAL_METHODS}
    print("RUN_TRAINING=False — bỏ qua train.")

## Evaluation — raw vs span-aligned EM/F1

**Quan trọng:** Phải load `base (Unsloth)` + `PeftModel.from_pretrained(adapter)` — không dùng `AutoModelForCausalLM` hay `FastLanguageModel.from_pretrained(adapter_path)` trực tiếp (sẽ eval base model gốc → 0% EM/F1).

Eval từng adapter trong `EVAL_METHODS`. Trước khi chạy full eval, cell sẽ in 3 sample debug để kiểm tra adapter đã gắn.

In [ ]:
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())

def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))

def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0: return 0.0
    p, r = n/len(pt), n/len(tt)
    return 2*p*r/(p+r)

def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip('"\'')
    pred = PREFIX_RE.sub("", pred).strip()
    return pred

def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx+len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i+1, min(i+n+4, len(words))+1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred


def _read_adapter_base_model(adapter_path):
    cfg_path = Path(adapter_path) / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Thiếu adapter_config.json trong {adapter_path}")
    cfg = json.load(open(cfg_path, encoding="utf-8"))
    return cfg.get("base_model_name_or_path", BASE_MODEL_NAME)


def _validate_adapter_files(adapter_path):
    """Kiểm tra adapter_model.safetensors tồn tại và đọc được."""
    adapter_path = Path(adapter_path)
    st_path = adapter_path / "adapter_model.safetensors"
    if not st_path.exists():
        raise FileNotFoundError(f"Thiếu {st_path}")

    head = open(st_path, "rb").read(200)
    if head.startswith(b"version https://git-lfs"):
        raise RuntimeError(
            f"{st_path.name} là Git LFS pointer — chưa pull weights thật. Chạy: git lfs pull"
        )

    size_mb = st_path.stat().st_size / (1024 * 1024)
    if size_mb < 0.1:
        raise RuntimeError(
            f"adapter_model.safetensors quá nhỏ ({size_mb:.2f} MB) — file corrupt hoặc train chưa save xong."
        )

    from safetensors import safe_open
    try:
        with safe_open(str(st_path), framework="pt", device="cpu") as f:
            _ = f.keys()
    except Exception as e:
        raise RuntimeError(f"{st_path.name} không đọc được: {e}") from e

    print(f"Adapter file OK: {st_path.name} ({size_mb:.1f} MB)")


def load_eval_model(adapter_path, max_seq_length):
    """Load base (Unsloth) + gắn LoRA adapter — load adapter trên CPU trước (fix SafetensorError)."""
    import torch
    from unsloth import FastLanguageModel
    from peft import PeftModel

    _validate_adapter_files(adapter_path)

    if LOAD_IN_4BIT or LOAD_IN_8BIT:
        load_dtype = None
    elif FORCE_FP16 or not torch.cuda.is_bf16_supported():
        load_dtype = torch.float16
    else:
        load_dtype = None

    base_model = _read_adapter_base_model(adapter_path)
    print(f"Eval load: base={base_model} | adapter={adapter_path}")

    attn_impl = resolve_attn_implementation()
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model,
            max_seq_length=max_seq_length,
            dtype=load_dtype,
            load_in_4bit=LOAD_IN_4BIT,
            load_in_8bit=LOAD_IN_8BIT,
            attn_implementation=attn_impl,
    )
    except Exception as e:
        if attn_impl != "sdpa":
            print(f"[WARN] eval attn={attn_impl} failed ({e}); fallback sdpa", flush=True)
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=base_model,
                max_seq_length=max_seq_length,
                dtype=load_dtype,
                load_in_4bit=LOAD_IN_4BIT,
                load_in_8bit=LOAD_IN_8BIT,
                attn_implementation="sdpa",
            )
        else:
            raise

    # Fix SafetensorError: load adapter weights on CPU first
    model = model.cpu()
    clear_gpu()
    model = PeftModel.from_pretrained(
        model,
        adapter_path,
        is_trainable=False,
        torch_device="cpu",
        low_cpu_mem_usage=True,
    )

    if not hasattr(model, "peft_config") or not model.peft_config:
        raise RuntimeError("LoRA adapter KHÔNG được gắn — eval sẽ ra 0% nếu tiếp tục.")
    print(f"Adapter OK: {list(model.peft_config.keys())}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    FastLanguageModel.for_inference(model)
    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer


def debug_eval_samples(model, tokenizer, samples, max_seq_length, n=3):
    """In vài prediction mẫu để phát hiện sớm eval load sai adapter."""
    print("\n--- Debug eval samples ---")
    for s in samples[:n]:
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model.device)
        with torch.no_grad():
            out = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred = clean_prediction(raw)
        print(f"Q: {s['question'][:80]}")
        print(f"GT: {s['answer']}")
        print(f"Pred(raw): {raw[:120]}")
        print(f"Pred(clean): {pred[:120]}")
        print("-" * 40)


def eval_one_adapter(method_name, adapter_path, test_samples, max_seq_length):
    print(f"\n--- Evaluating: {method_name} | {adapter_path} ---")
    if not Path(adapter_path).exists():
        print(f"SKIP {method_name}: adapter path không tồn tại.")
        return None

    clear_gpu()
    model_eval, tokenizer_eval = load_eval_model(adapter_path, max_seq_length)
    debug_eval_samples(model_eval, tokenizer_eval, test_samples, max_seq_length, n=3)

    model_eval.generation_config.max_length = None
    model_eval.generation_config.max_new_tokens = MAX_NEW_TOKENS

    has_em_r, has_f1_r, has_em_a, has_f1_a, no_ok, preds = [], [], [], [], [], []
    total = len(test_samples)
    pbar = tqdm(test_samples, total=total, desc=f"Eval-{method_name}", unit="sample", bar_format=TQDM_BAR)

    for i, s in enumerate(pbar, 1):
        pbar.set_description(f"Eval-{method_name} {i}/{total}")
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(prompt, return_tensors="pt", truncation=True, max_length=max_seq_length).to(model_eval.device)
        with torch.no_grad():
            out = model_eval.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )
        raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, s["context"]) if USE_SPAN_POSTPROCESS else pred_raw

        if s["is_impossible"]:
            ok = int(is_no_answer(pred))
            no_ok.append(ok)
            em_r = em_a = ok
            f1_r = f1_a = float(ok)
        else:
            gt = s["answer"]
            em_r = compute_em(pred_raw, gt)
            f1_r = compute_f1(pred_raw, gt)
            em_a = compute_em(pred, gt)
            f1_a = compute_f1(pred, gt)
            has_em_r.append(em_r)
            has_f1_r.append(f1_r)
            has_em_a.append(em_a)
            has_f1_a.append(f1_a)

        preds.append({
            "id": s["id"], "question": s["question"], "is_impossible": s["is_impossible"],
            "ground_truth": s["answer"], "prediction_raw": pred_raw, "prediction": pred,
            "em_raw": em_r if not s["is_impossible"] else em_a,
            "f1_raw": f1_r if not s["is_impossible"] else f1_a,
            "em": em_a, "f1": f1_a,
        })
        if has_em_a:
            pbar.set_postfix(has_em=f"{100*sum(has_em_a)/len(has_em_a):.1f}%")

    n_has, n_no = len(has_em_a), len(no_ok)
    metrics = {
        "method": method_name,
        "adapter": adapter_path,
        "hasans_em_raw": round(100*sum(has_em_r)/max(n_has, 1), 4),
        "hasans_f1_raw": round(100*sum(has_f1_r)/max(n_has, 1), 4),
        "hasans_em": round(100*sum(has_em_a)/max(n_has, 1), 4),
        "hasans_f1": round(100*sum(has_f1_a)/max(n_has, 1), 4),
        "noans_accuracy": round(100*sum(no_ok)/n_no, 4) if n_no else None,
        "n_hasans": n_has, "n_noans": n_no, "total": total,
    }

    del model_eval, tokenizer_eval
    clear_gpu()
    return {"metrics": metrics, "predictions": preds}


In [ ]:
if RUN_EVALUATION:
    import warnings; warnings.filterwarnings("ignore")

    if "trained_paths" not in globals():
        trained_paths = {m: get_variant(m)["save_path"] for m in EVAL_METHODS}

    all_results = {}
    for method in EVAL_METHODS:
        adapter_path = trained_paths.get(method) or get_variant(method)["save_path"]
        result = eval_one_adapter(method, adapter_path, test_samples, max_seq_length)
        if result is not None:
            all_results[method] = result

    if not all_results:
        print("Không có adapter nào để eval.")
    else:
        line = "=" * 60
        print("\n" + line)
        print("   KẾT QUẢ ĐÁNH GIÁ - UIT-ViQuAD2.0 (Qwen2.5-3B)")
        print(line)
        print(f"{'Method':<10} {'EM(raw)':>9} {'F1(raw)':>9} {'EM':>9} {'F1':>9} {'NoAns':>9}")
        print("-" * 60)
        for method, result in all_results.items():
            m = result["metrics"]
            noans = f"{m['noans_accuracy']:.1f}%" if m["noans_accuracy"] is not None else "N/A"
            print(
                f"{method:<10} {m['hasans_em_raw']:>8.2f}% {m['hasans_f1_raw']:>8.2f}% "
                f"{m['hasans_em']:>8.2f}% {m['hasans_f1']:>8.2f}% {noans:>9}"
            )
        print(line)

        if len(all_results) == 1:
            method = next(iter(all_results))
            out_path = EVAL_RESULTS_PATH
            payload = {
                "dataset": DATASET_NAME,
                "base_model": BASE_MODEL_NAME,
                "max_seq_length": max_seq_length,
                "use_span_postprocess": USE_SPAN_POSTPROCESS,
                **all_results[method]["metrics"],
                "predictions": all_results[method]["predictions"],
            }
        else:
            out_path = COMPARE_EVAL_PATH
            payload = {
                "dataset": DATASET_NAME,
                "base_model": BASE_MODEL_NAME,
                "max_seq_length": max_seq_length,
                "use_span_postprocess": USE_SPAN_POSTPROCESS,
                "methods": {k: v["metrics"] for k, v in all_results.items()},
                "predictions": {k: v["predictions"] for k, v in all_results.items()},
            }

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        print(f"Saved → {out_path}")